<a href="https://colab.research.google.com/github/isumakm/Weather-Prediction-and-Crop-Recommendation-System-/blob/Multi-Crop-Ranking/Multi_Crop_Ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Setup**

In [35]:
!pip -q install lightgbm

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

##**Load Data**

In [ ]:
try:
    df_req = pd.read_csv("/content/drive/MyDrive/DSGP/Crop_training_data_FULL_final.csv")
    print("Dataset loaded correctly")
    print(df_req.shape)

except Exception as e:
    print("Dataset loading failed:", e)

df_req.head()

Dataset loaded correctly
(2100, 14)


,crop,temperature,rainfall,sunshine_hours,ph,organic_carbon,cec,awc,bulk_density,rooting_depth_m,texture,texture_code,suitability,suitability_class
0,Brinjal,17.272137,2734.849798,9.257206,7.098268,1.881237,33.767915,0.389893,0.945723,1.471861,silt loam,5,0.698,Unsuitable
1,Brinjal,23.442325,2079.442081,3.802854,7.902331,2.747803,34.757119,0.105033,1.655120,1.323896,sand,1,0.589,Unsuitable
2,Brinjal,35.281442,1567.821919,4.822195,5.368965,1.014584,6.906058,0.343163,1.050276,1.157586,loamy sand,2,0.724,Unsuitable
3,Brinjal,29.140985,985.385576,8.182605,7.006299,3.335882,23.568867,0.393432,1.408769,0.542888,loamy sand,2,0.860,Suitable
4,Brinjal,21.023598,815.960211,5.464674,7.233649,2.836555,34.949356,0.165123,1.406248,0.640387,silt loam,5,0.835,Suitable


##**Required Features**

In [ ]:
# Keep one row per crop as the requirement profile
feature_req_cols = [
    "temperature", "rainfall", "sunshine_hours",
    "ph", "organic_carbon", "cec", "awc", "bulk_density",
    "rooting_depth_m", "texture_code"
]

df_crop_profile = df_req.groupby("crop")[feature_req_cols].mean().reset_index()

crops = df_crop_profile["crop"].tolist()
print("Crops:", len(crops))
df_crop_profile.head()

Crops: 21


,crop,temperature,rainfall,sunshine_hours,ph,organic_carbon,cec,awc,bulk_density,rooting_depth_m,texture_code
0,Banana,27.548190,1657.514772,6.299725,6.338439,2.424694,21.067627,0.274485,1.298976,0.872142,4.24
1,Bitter Gourd,28.090295,1525.525487,6.310824,6.341706,2.286389,22.145218,0.284207,1.314739,0.917512,4.04
2,Brinjal,27.067977,1562.838623,6.328381,6.101815,2.043603,23.542169,0.276034,1.314154,0.933927,3.64
3,Capsicum,28.692649,1664.949702,6.740908,6.185201,2.277045,23.084237,0.274618,1.262559,0.872491,4.31
4,Cucumber,27.663707,1619.559030,6.263031,6.258471,2.264791,22.515534,0.275996,1.285415,0.820276,4.18


##**Suitability scoring function**

In [ ]:
# 2) Define a smooth suitability scoring function
#    (score 1 inside range; smoothly decays outside)
# =========================
def smooth_score(x, lo, hi):
    """
    1 if within [lo,hi]
    Smooth decay outside using relative distance
    """
    if lo <= x <= hi:
        return 1.0
    # distance outside
    if x < lo:
        return max(0.0, 1.0 - (lo - x) / (abs(lo) + 1e-6))
    if x > hi:
        return max(0.0, 1.0 - (x - hi) / (abs(hi) + 1e-6))

def compute_suitability(env, crop_row, weights=None):
    """
    env: dict of farmer context features
    crop_row: crop requirement profile (one row from df_crop_profile)
    weights: dict feature->weight (optional)
    returns suitability in [0,1]
    """

    # default weights (edit if you want)
    if weights is None:
        weights = {
            "temperature": 0.22,
            "rainfall": 0.22,
            "sunshine_hours": 0.12,
            "ph": 0.12,
            "organic_carbon": 0.08,
            "cec": 0.08,
            "awc": 0.08,
            "bulk_density": 0.05,
            "rooting_depth_m": 0.02,
            "texture_code": 0.01
        }

    # we create a "preferred band" around the crop profile mean
    # These tolerances are reasonable defaults (you can tune later)
    tol = {
        "temperature": 3.0,
        "rainfall": 150.0,
        "sunshine_hours": 1.5,
        "ph": 0.4,
        "organic_carbon": 0.6,
        "cec": 4.0,
        "awc": 0.05,
        "bulk_density": 0.12,
        "rooting_depth_m": 0.25,
        "texture_code": 1.0
    }

    total = 0.0
    wsum = 0.0

    for f, w in weights.items():
        c = float(crop_row[f])
        t = float(tol[f])

        lo, hi = c - t, c + t
        s = smooth_score(float(env[f]), lo, hi)
        total += w * s
        wsum += w

    return total / (wsum + 1e-9)

In [ ]:
# 3) Generate correct ranking training data (queries)
#    Input to your system: location + planting date (NO season length)
#    - Weather + soil models will output env features
#    Here we simulate those outputs to create training rows.
# =========================
def sample_farmer_context():
    """
    Simulate outputs from soil+weather models for one farmer context.
    Replace these ranges later with real model output ranges.
    """
    return {
        "temperature": np.random.uniform(15, 38),
        "rainfall": np.random.uniform(200, 2500),
        "sunshine_hours": np.random.uniform(3, 9),
        "ph": np.random.uniform(4.5, 8.0),
        "organic_carbon": np.random.uniform(0.5, 4.0),
        "cec": np.random.uniform(5, 40),
        "awc": np.random.uniform(0.10, 0.45),
        "bulk_density": np.random.uniform(0.9, 1.7),
        "rooting_depth_m": np.random.uniform(0.3, 2.0),
        "texture_code": np.random.uniform(1, 7)
    }

N_QUERIES = 5000   # farmer contexts
rows = []

for q in range(N_QUERIES):
    env = sample_farmer_context()

    # each query must include multiple crops ( ideally all crops)
    for _, crop_row in df_crop_profile.iterrows():
        score_val = compute_suitability(env, crop_row)

        row_data = {
            "query_id": q,
            "crop": crop_row["crop"],
            "label": score_val,
            **{f"env_{k}": v for k, v in env.items()},
            **{f"crop_{k}": v for k, v in crop_row[feature_req_cols].items()}
        }
        rows.append(row_data)

df_rank = pd.DataFrame(rows)
print(df_rank.shape)
df_rank.head()

# Convert label to integer relevance
# We map suitability into 0..4 (5 levels)
df_rank["relevance"] = pd.cut(
    df_rank["label"],
    bins=[-1, 0.3, 0.5, 0.7, 0.85, 1.01],
    labels=[0, 1, 2, 3, 4]
).astype(int)

(105000, 23)


##**Train/Test split by query**

In [ ]:
env_feature_cols = [f"env_{col}" for col in feature_req_cols]
crop_feature_cols = [f"crop_{col}" for col in feature_req_cols]
feature_cols = env_feature_cols + crop_feature_cols

X = df_rank[feature_cols]
y = df_rank["relevance"]
groups = df_rank["query_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df_rank.iloc[train_idx].copy()
test_df  = df_rank.iloc[test_idx].copy()

print("Train queries:", train_df["query_id"].nunique())
print("Test queries :", test_df["query_id"].nunique())

# Create group sizes for LightGBM ranker
train_group = train_df.groupby("query_id").size().values
test_group  = test_df.groupby("query_id").size().values

X_train, y_train = train_df[feature_cols], train_df["relevance"]
X_test, y_test   = test_df[feature_cols], test_df["relevance"]

Train queries: 4000
Test queries : 1000


##**Train LambdaMART**

In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED
)

ranker.fit(
    X_train, y_train,
    group=train_group,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group],
    eval_at=[5]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014850 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 84000, number of used features: 10


LGBMRanker(colsample_bytree=0.8, learning_rate=0.05, metric='ndcg',
           n_estimators=500, objective='lambdarank', random_state=42,
           subsample=0.8)

##**Accuracy check**

In [ ]:
def ndcg_at_k(df_part, k=5):
    """
    Compute mean NDCG@k across queries
    """
    scores = []
    for qid, g in df_part.groupby("query_id"):
        true_rel = g["relevance"].values.reshape(1, -1)
        pred = ranker.predict(g[feature_cols]).reshape(1, -1)
        scores.append(ndcg_score(true_rel, pred, k=k))
    return float(np.mean(scores))

train_ndcg5 = ndcg_at_k(train_df, k=5)
test_ndcg5  = ndcg_at_k(test_df, k=5)

print(f"Train NDCG@5: {train_ndcg5:.4f}")
print(f"Test  NDCG@5: {test_ndcg5:.4f}")

# Overfitting signal:
# If train NDCG@5 is much higher than test, model is overfitting.
print("Overfitting gap (train-test):", round(train_ndcg5 - test_ndcg5, 4))

Train NDCG@5: 0.9701
Test  NDCG@5: 0.9671
Overfitting gap (train-test): 0.0029


##**Example**

In [ ]:
# Use the model to rank crops for ONE farmer context
def recommend_top_k(env, k=5):
    rows = []
    for _, crop_row in df_crop_profile.iterrows():
        row_data = {
            "crop": crop_row["crop"],
            **{f"env_{k}": v for k, v in env.items()},
            **{f"crop_{k}": v for k, v in crop_row[feature_req_cols].items()}
        }
        rows.append(row_data)

    df_in = pd.DataFrame(rows)
    preds = ranker.predict(df_in[feature_cols])
    df_in["rank_score"] = preds
    df_in = df_in.sort_values("rank_score", ascending=False)
    return df_in[["crop", "rank_score"]].head(k)

example_env = sample_farmer_context()
recommend_top_k(example_env, k=5)

,crop,rank_score
0,Banana,-5.922690e-08
1,Bitter Gourd,-5.922690e-08
2,Brinjal,-5.922690e-08
3,Capsicum,-5.922690e-08
4,Cucumber,-5.922690e-08
